V11 Мат модель для нахождения коэффициента влияния гендера на ЗП

In [18]:
import pandas as pd

modelv1 = pd.read_excel("../../data/processed/v1_clean_base.xlsx")

model v11 (логарифмическая регрессия по параметрам: salary, gender, experience, education_type, region_code, industry_code)

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

cols = ["salary", "gender", "experience", "education_type", "region_code", "industry_code"]

data = modelv1[cols].dropna().copy()
data = data[data["salary"] > 0].copy()
data["log_salary"] = np.log(data["salary"])

min_n = 20
results = []

for (region, industry), g in data.groupby(["region_code", "industry_code"]):
    if len(g) < min_n:
        continue

    if g["gender"].nunique() < 2:
        continue

    model = smf.ols(
        "log_salary ~ C(gender) + experience + C(education_type)",
        data=g
    ).fit()

    for term in model.params.index:
        if "C(gender)" in term:
            results.append({
                "region_code": region,
                "industry_code": industry,
                "n": len(g),
                "term": term,
                "coef": model.params[term],
                "pvalue": model.pvalues[term],
                "r2": model.rsquared
            })

v11_result = pd.DataFrame(results).sort_values(["region_code", "industry_code"])

# Проверка данных

print(v11_result.head())

   region_code industry_code   n                  term      coef    pvalue  \
0            2      DeskWork  83  C(gender)[T.Мужской]  0.344403  0.041464   
1            2     Education  34  C(gender)[T.Мужской] -0.291378  0.103497   
2            2      Industry  21  C(gender)[T.Мужской]  0.280312  0.101629   
3            2      Medicine  20  C(gender)[T.Мужской] -0.046181  0.809440   
4            2         Sales  40  C(gender)[T.Мужской]  0.057291  0.812803   

         r2  
0  0.169555  
1  0.424176  
2  0.196310  
3  0.551253  
4  0.318462  


Сохранение результатов model v11

In [9]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

cols = ["salary", "gender", "experience", "education_type", "region_code", "industry_code"]

data = modelv1[cols].dropna().copy()
data = data[data["salary"] > 0].copy()
data["log_salary"] = np.log(data["salary"])

min_n = 20
results = []

for region, g in data.groupby("region_code"):
    if len(g) < min_n:
        continue

    if g["gender"].nunique() < 2:
        continue

    model = smf.ols(
        "log_salary ~ C(gender) + experience + C(education_type)",
        data=g
    ).fit()

    for name, coef in model.params.items():
        if "C(gender)" in name:
            results.append({
                "region_code": region,
                "n": len(g),
                "term": name,
                "coef": coef,
                "pvalue": model.pvalues[name],
                "r2": model.rsquared,
                "pct_effect": (np.exp(coef) - 1) * 100
            })

v11_result_wout_reg = pd.DataFrame(results).sort_values("region_code")

print(v11_result_wout_reg.head())

   region_code    n                  term      coef    pvalue        r2  \
0            1  102  C(gender)[T.Мужской]  0.035948  0.669823  0.102967   
1            2  382  C(gender)[T.Мужской]  0.216975  0.000005  0.167314   
2            3  102  C(gender)[T.Мужской]  0.290665  0.000378  0.325877   
3            4   27  C(gender)[T.Мужской]  0.272641  0.243648  0.177865   
4            5  304  C(gender)[T.Мужской]  0.116092  0.040387  0.038214   

   pct_effect  
0    3.660193  
1   24.231283  
2   33.731673  
3   31.342904  
4   12.309912  


In [10]:
v11_result_wout_reg.to_excel("../../data/processed/v11_result_wout_reg.xlsx", index=False)